In [1]:
!pip install plotly

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.6/15.6 MB 31.8 MB/s eta 0:00:0000:0100:01


In [2]:
!pip install "anywidget>=0.9.13"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 213.7/213.7 kB 1.5 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.8/139.8 kB 2.6 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 477.3/477.3 kB 5.1 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 14.6 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 914.9/914.9 kB 10.4 MB/s eta 0:00:0000:01


In [3]:
!pip install -U kaleido

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.9/79.9 MB 12.5 MB/s eta 0:00:0000:0100:01


In [4]:
!pip install anndata==0.8.0

In [1]:
def gexpnorm(adata, gene_dict,level, gl, fin_name):
    gi = [gene_dict[i] for i in gl]
    exp = adata[:,gi].X.A
    expr = pd.DataFrame(exp,columns = gl, index = adata.obs_names)
    expr["cell_type"] = adata.obs[level].values
    mean_expr = expr.groupby("cell_type").mean()
    frac_expr = expr.groupby("cell_type").apply(
    lambda x: (x > 0).mean())
    mean_expr_norm = (mean_expr - mean_expr.min()) / (
    mean_expr.max() - mean_expr.min())
    mean_expr.columns = fin_name
    frac_expr.columns = fin_name
    mean_expr_norm.columns = fin_name
    return(mean_expr_norm, frac_expr)

In [2]:
def orthogroup_mapper(orthogroups, label):
    mapping = {}
    for index in orthogroups.index:
        gene_list = orthogroups.loc[index, label]
        if type(gene_list) != float:
            genes = gene_list.split(',')
            for item in genes:
                mapping[item] = index
    return(mapping)

In [3]:
from samap.mapping import SAMAP
from samap.analysis import (get_mapping_scores, GenePairFinder, transfer_annotations,
                            sankey_plot, chord_plot, CellTypeTriangles, 
                            ParalogSubstitutions, FunctionalEnrichment,
                            convert_eggnog_to_homologs, GeneTriangles)
from samalg import SAM
import pandas as pd
from Bio import SeqIO
from samap.utils import (save_samap, load_samap)
import scanpy as sc
import matplotlib.colors
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats
from scipy import sparse 
from scipy import cluster
import seaborn as sns
import random
import sklearn
from scipy.stats import poisson
from sklearn.neighbors import KernelDensity
import time
import dill
from scipy.optimize import minimize
import pickle
import itertools
import os
import anndata as ad
import math
import csv
import plotly.express as px
from tqdm import tqdm
from collections import Counter

In [4]:
fn = 'Active_SAM_joined/SAM_CJ_joined_v2_cleaned_03122025.h5ad'
sam_cj = SAM()
sam_cj.load_data(fn)
gene_dict_cj = {}
for i in range(len(sam_cj.adata.var_names)):
    gene_dict_cj[sam_cj.adata.var_names[i]] = i
gene_dict_cj['NaN'] = 'NaN'

In [5]:
fn = 'Active_SAM_joined/SAM_AC_ncbi_soupx_cleaned_03122025.h5ad'
sam_ac = SAM()
sam_ac.load_data(fn)
gene_dict_ac = {}
for i in range(len(sam_ac.adata.var_names)):
    gene_dict_ac[sam_ac.adata.var_names[i]] = i
gene_dict_ac['NaN'] = 'NaN'

In [6]:
fn = 'Active_SAM_joined/SAM_XT_joined_Slc17a6_cleaned_03122205.h5ad'
sam_xt = SAM()
sam_xt.load_data(fn)
gene_dict_xt = {}
for i in range(len(sam_xt.adata.var_names)):
    gene_dict_xt[sam_xt.adata.var_names[i]] = i
gene_dict_xt['NaN'] = 'NaN'

In [7]:
fn = 'Active_SAM_joined/SAM_DR_ncbi_joined_cleaned_07172026.h5ad'
sam_dr = SAM()
sam_dr.load_data(fn)
gene_dict_dr = {}
for i in range(len(sam_dr.adata.var_names)):
    gene_dict_dr[sam_dr.adata.var_names[i]] = i
gene_dict_dr['NaN'] = 'NaN'

In [8]:
orthogroups = pd.read_csv('Vert_emapper_allorgs_07032026.tsv',delimiter='\t', index_col = 'Unnamed: 0')
mg_mapping_ortho = orthogroup_mapper(orthogroups, 'MM')
mo_mapping_ortho = orthogroup_mapper(orthogroups, 'MO')
cj_mapping_ortho = orthogroup_mapper(orthogroups, 'CJ')
ac_mapping_ortho = orthogroup_mapper(orthogroups, 'AC')
xt_mapping_ortho = orthogroup_mapper(orthogroups, 'XT')
dr_mapping_ortho = orthogroup_mapper(orthogroups,'DR')

In [12]:
tested_markers = {'cj_xt_1':['Trpc4','Trpc6'],
'cj_xt_2':['Tpo','Trhr','Drd5'],
'cj_xt_3':['Cyp26a1','Kiss1r'],
'cj_xt_5':['Sgcg','Drd1','Igf1','Cck'],
'cj_xt_dr_1':['Jcad','Fgd6','Slc1a6','Rps6ka2'],
'cj_ac_xt_dr_1':['Drd5','Ndnf','Crh','Trpc6','Nmur1','Mtnr1a','Nmu','Ddc','Th'],
'cj_ac_xt_dr_3':['Ngf'],
'cj_xt_dr_2':['Ddc','Tph1','Opn5'],
'cj_ac_1':[],
'cj_xt_4':['Brs3','Gabrd','Nmur1','Tnik','Rps6ka2'],
'cj_ac_xt_1':['Gnrh2','Trpm2'],
'xt_dr_1':['Col1a2','Slc8a3'],
'cj_ac_xt_dr_2':['Crh','Chrnb3','Npy2r','Trh'],
'cj_ac_xt_dr_4':['Prokr1'],
'extra_genes':[]}

mg_list = []
for item in tested_markers:
    mg_list = mg_list + tested_markers[item]

In [54]:
tested_markers = {'cj_ac_xt_dr_4':['Lhx2','Lhx9','Tcf7l2'], 
                  'cj_ac_xt_dr_3':['Gata3','Tal1'],
                  'cj_ac_xt_dr_1':['Ebf3','Irx1','Lef1'],
                  'cj_xt_dr_2':['Prdm16', 'Zic1', 'Zic4'],
                  'cj_ac_xt_1':['Pitx2', 'Sim1', 'Uncx'],
                  'cj_xt_dr_1':['Prdm13','Sp5','St18'],
                  'ac_xt_3':['Casz1', 'Esrrg', 'Npas3'],
                  'cj_ac_1':['Meis2', 'Nr2f2', 'Tshz2'],
                  'ac_xt_4':['Bhlhe22', 'Foxg1','Tcf4'],
                  'ac_xt_2':['Insm1','Nr2e1','Zfp516'],
                  'cj_ac_2':['Prox1','Nr3c2','Rora'],
                  'ac_xt_1':['Bsx','Esr1','Hmx2'],
                  'ac_dr_1':['Neurod2','Pitx2','Satb2'],
                  'cj_ac_3':['Tfap2a','Otx2','Npas3'],
                  'cj_ac_4':['Tfap2a','Barhl2','Zeb2']}

mg_list = []
for item in tested_markers:
    mg_list = mg_list + tested_markers[item]

In [55]:
def remove_latter_duplicates(lst):
    seen = set()
    result = []
    for item in lst:
        if item not in seen:
            seen.add(item)
            result.append(item)
    return result

In [56]:
mg_list_cl = remove_latter_duplicates(mg_list)

In [57]:
def check_repeats(lst):
    counts = Counter(lst)
    repeats = {val: cnt for val, cnt in counts.items() if cnt > 1}
    
    if repeats:
        print(f"Found {len(repeats)} repeated value(s):")
        for val, cnt in repeats.items():
            print(f"  '{val}' appears {cnt} times")
    else:
        print("No repeated values found.")

    return repeats

In [58]:
check_repeats(mg_list_cl)

No repeated values found.


{}

In [59]:
mg_list = mg_list_cl

In [60]:
mg_list

['Lhx2',
 'Lhx9',
 'Tcf7l2',
 'Gata3',
 'Tal1',
 'Ebf3',
 'Irx1',
 'Lef1',
 'Prdm16',
 'Zic1',
 'Zic4',
 'Pitx2',
 'Sim1',
 'Uncx',
 'Prdm13',
 'Sp5',
 'St18',
 'Casz1',
 'Esrrg',
 'Npas3',
 'Meis2',
 'Nr2f2',
 'Tshz2',
 'Bhlhe22',
 'Foxg1',
 'Tcf4',
 'Insm1',
 'Nr2e1',
 'Zfp516',
 'Prox1',
 'Nr3c2',
 'Rora',
 'Bsx',
 'Esr1',
 'Hmx2',
 'Neurod2',
 'Satb2',
 'Tfap2a',
 'Otx2',
 'Barhl2',
 'Zeb2']

In [15]:
otohomo = pd.read_csv('OTO_star_nothreshold_missing_le2_expressionthresh_07062026.tsv',delimiter='\t',index_col = 'MM')

In [53]:
otohomo.loc['Gnrh2',:] = [np.nan,np.nan,'LOC134299345','gnrh2','gnrh2',3]

In [54]:
otohomo.loc['Lhx6',:] = [np.nan,'LHX6','lhx6','lhx6','lhx6a',3]

In [61]:
CJ_names = {i:otohomo.loc[i,'CJ'] for i in mg_list}
AC_names = {i:otohomo.loc[i,'AC'] for i in mg_list}
XT_names = {i:otohomo.loc[i,'XT'] for i in mg_list}
DR_names = {i:otohomo.loc[i,'DR'] for i in mg_list}

In [62]:
cl_CJ_names = {k:v for k,v in CJ_names.items() if not pd.isna(v) and v in gene_dict_cj}
cl_AC_names = {k:v for k,v in AC_names.items() if not pd.isna(v) and v in gene_dict_ac}
cl_XT_names = {k:v for k,v in XT_names.items() if not pd.isna(v) and v in gene_dict_xt}
cl_DR_names = {k:v for k,v in DR_names.items() if not pd.isna(v) and v in gene_dict_dr}

In [63]:
cl_CJ_names

{'Lhx2': 'LHX2',
 'Lhx9': 'LHX9',
 'Tcf7l2': 'TCF7L2',
 'Gata3': 'GATA3',
 'Tal1': 'TAL1',
 'Ebf3': 'EBF3',
 'Irx1': 'IRX1',
 'Lef1': 'LEF1',
 'Prdm16': 'PRDM16',
 'Zic1': 'ZIC1',
 'Zic4': 'ZIC4',
 'Pitx2': 'ENSCJPG00005006843',
 'Sim1': 'SIM1',
 'Uncx': 'UNCX',
 'Prdm13': 'PRDM13',
 'Sp5': 'SP5',
 'St18': 'ST18',
 'Esrrg': 'ESRRG',
 'Npas3': 'NPAS3',
 'Meis2': 'ENSCJPG00005017524',
 'Nr2f2': 'NR2F2',
 'Tshz2': 'TSHZ2',
 'Foxg1': 'FOXG1',
 'Tcf4': 'ENSCJPG00005004819',
 'Nr2e1': 'NR2E1',
 'Zfp516': 'ZNF516',
 'Prox1': 'PROX1',
 'Nr3c2': 'NR3C2',
 'Rora': 'RORA',
 'Esr1': 'ESR1',
 'Hmx2': 'HMX2',
 'Neurod2': 'NEUROD2',
 'Satb2': 'SATB2',
 'Tfap2a': 'TFAP2A',
 'Otx2': 'OTX2',
 'Barhl2': 'BARHL2',
 'Zeb2': 'ZEB2'}

In [64]:
index_order = tested_markers.keys()

In [65]:
len(tested_markers.keys())

15

In [66]:
sam_cj.adata.obs.columns

Index(['orig.ident', 'nCount_RNA', 'nFeature_RNA', 'n_genes', 'n_counts',
       'key', 'hicat_merged', 'subclass_id_label_mapping',
       'subclass_id_label_lc', 'leiden_clusters',
       'subclass_id_label_mapping_nounlabeled', 'neurotransmitter',
       'region_label', 'subclass_id_label_reduced_mapping',
       'subclass_id_label_reduced_lc',
       'subclass_id_label_reduced_mapping_nounlabeled', 'fraction_match',
       'best_match', 'frac_match_test', 'leiden_removal', 'nCount_SCT',
       'nFeature_SCT', 'SCT_snn_res.0.8', 'seurat_clusters', 'SCT_snn_res.5',
       'eq_subclass', 'eq_subclass_lc', 'eq_subclass_frac',
       'eq_subclass_nounlabeled', 'eq_subclass_nounlabeled_NN',
       'eq_subclass_nounlabeled_nmm', 'ss_subclass', 'ss_subclass_nounlabeled',
       'ss_class', 'ss_subclass_nounlabeled_astro', 'ss_subclass_v2',
       'ss_subclass_v2_nounlabeled', 'ss_subclass_nounlabeled_nmm',
       'ss_subclass_v3_nounlabeled', 'subclass_id_label_crossed',
       'ss_subclas

In [67]:
nonmam_cj = sam_cj.adata[sam_cj.adata.obs['ss_subclass_nounlabeled_nmm_cl_v4_nn'].isin(index_order)]
nonmam_ac = sam_ac.adata[sam_ac.adata.obs['ss_subclass_nounlabeled_nmm_cl_v4_nn'].isin(index_order)]
nonmam_xt = sam_xt.adata[sam_xt.adata.obs['ss_subclass_nounlabeled_nmm_cl_v4_nn'].isin(index_order)]
nonmam_dr = sam_dr.adata[sam_dr.adata.obs['ss_subclass_nounlabeled_nmm_cl_v4_nn'].isin(index_order)]

In [89]:
mean_expr_norm_cj, frac_cj = gexpnorm(nonmam_cj,gene_dict_cj,'ss_subclass_nounlabeled_nmm_cl_v4_nn',
                             [cl_CJ_names[i] for i in mg_list if i in cl_CJ_names],
                            [i for i in mg_list if i in cl_CJ_names])
mean_expr_norm_ac, frac_ac = gexpnorm(nonmam_ac,gene_dict_ac,'ss_subclass_nounlabeled_nmm_cl_v4_nn',
                             [cl_AC_names[i] for i in mg_list if i in cl_AC_names],
                            [i for i in mg_list if i in cl_AC_names])
mean_expr_norm_xt, frac_xt = gexpnorm(nonmam_xt,gene_dict_xt,'ss_subclass_nounlabeled_nmm_cl_v4_nn',
                             [cl_XT_names[i] for i in mg_list if i in cl_XT_names],
                            [i for i in mg_list if i in cl_XT_names])
mean_expr_norm_dr, frac_dr = gexpnorm(nonmam_dr,gene_dict_dr,'ss_subclass_nounlabeled_nmm_cl_v4_nn',
                             [cl_DR_names[i] for i in mg_list if i in cl_DR_names],
                            [i for i in mg_list if i in cl_DR_names])

In [90]:
mean_expr_norm_cj

,Lhx2,Lhx9,Tcf7l2,Gata3,Tal1,Ebf3,Irx1,Lef1,Prdm16,Zic1,...,Nr3c2,Rora,Esr1,Hmx2,Neurod2,Satb2,Tfap2a,Otx2,Barhl2,Zeb2
cell_type,,,,,,,,,,,,,,,,,,,,,
cj_ac_1,0.008496,0.008149,0.000000,0.003500,0.000000,0.022062,0.104586,0.000000,0.025276,0.057567,...,0.054273,0.263210,1.000000,0.000000,NaN,0.112227,0.000000,0.007851,0.000000,0.019359
cj_ac_2,0.706006,0.732827,1.000000,0.000000,0.000000,0.035239,0.000000,0.129844,0.000000,1.000000,...,1.000000,0.824163,0.000000,0.000000,NaN,0.000000,0.000000,0.000000,0.000000,0.029155
cj_ac_3,0.000000,0.000000,0.006864,1.000000,0.766034,0.000000,0.000000,0.643972,0.006414,0.009991,...,0.085367,0.593676,0.025824,0.000000,NaN,0.000000,1.000000,0.930010,0.000000,1.000000
cj_ac_4,0.604834,0.553810,0.140147,0.007906,0.000000,1.000000,0.082914,0.667077,0.012407,0.019764,...,0.107383,0.665874,0.088560,0.000000,NaN,0.564126,0.805960,0.290440,1.000000,0.644481
cj_ac_xt_1,0.021479,0.008184,0.000500,0.003353,0.000000,0.219715,0.069107,0.002242,0.012235,0.026963,...,0.192009,0.339574,0.957819,0.162687,NaN,0.094468,0.005575,0.254677,0.671827,0.035396
cj_ac_xt_dr_1,0.902634,0.994570,0.569507,0.001746,0.022962,0.908865,1.000000,0.336258,0.012688,0.090979,...,0.226082,0.505757,0.170437,0.104150,NaN,0.178993,0.142389,0.110876,0.414745,0.276786
cj_ac_xt_dr_3,0.006692,0.027125,0.524730,0.393175,1.000000,0.205178,0.315232,0.070860,0.017371,0.068616,...,0.264727,0.387789,0.169285,0.271163,NaN,0.207969,0.459927,1.000000,0.034430,0.282078
cj_ac_xt_dr_4,1.000000,1.000000,0.910246,0.000000,0.000000,0.022609,0.000000,0.018697,0.054681,0.023254,...,0.556875,1.000000,0.024638,0.417677,NaN,0.385686,0.000000,0.034704,0.072702,0.022177
cj_xt_dr_1,0.000000,0.047614,0.567855,0.000000,0.000000,0.005723,0.000000,1.000000,0.060551,0.000000,...,0.000000,0.100874,0.198615,1.000000,NaN,0.163189,0.000000,0.065216,0.091319,0.000000


In [91]:
mean_expr_norm_cj = mean_expr_norm_cj.loc[[i for i in index_order if i in sam_cj.adata.obs['ss_subclass_nounlabeled_nmm_cl_v4_nn'].unique()],:]
mean_expr_norm_ac = mean_expr_norm_ac.loc[[i for i in index_order if i in sam_ac.adata.obs['ss_subclass_nounlabeled_nmm_cl_v4_nn'].unique()],:]
mean_expr_norm_xt = mean_expr_norm_xt.loc[[i for i in index_order if i in sam_xt.adata.obs['ss_subclass_nounlabeled_nmm_cl_v4_nn'].unique()],:]
mean_expr_norm_dr = mean_expr_norm_dr.loc[[i for i in index_order if i in sam_dr.adata.obs['ss_subclass_nounlabeled_nmm_cl_v4_nn'].unique()],:]

In [92]:
mean_expr_norm_ac.columns

Index(['Lhx2', 'Lhx9', 'Tcf7l2', 'Gata3', 'Tal1', 'Ebf3', 'Irx1', 'Lef1',
       'Prdm16', 'Zic1', 'Zic4', 'Pitx2', 'Sim1', 'Uncx', 'Prdm13', 'Sp5',
       'St18', 'Casz1', 'Esrrg', 'Npas3', 'Meis2', 'Nr2f2', 'Tshz2', 'Bhlhe22',
       'Foxg1', 'Tcf4', 'Insm1', 'Nr2e1', 'Zfp516', 'Prox1', 'Nr3c2', 'Rora',
       'Bsx', 'Esr1', 'Hmx2', 'Neurod2', 'Satb2', 'Tfap2a', 'Otx2', 'Barhl2',
       'Zeb2'],
      dtype='object')

In [93]:
df = pd.DataFrame(columns = ['celltype','gene','avg exp','frac'])

In [94]:
cjl = []
for ct in mean_expr_norm_cj.index:
    for g in mean_expr_norm_cj.columns:
        cjl.append(['cj_'+ct,g,mean_expr_norm_cj.loc[ct,g],frac_cj.loc[ct,g]])
        
acl = []
for ct in mean_expr_norm_ac.index:
    for g in mean_expr_norm_ac.columns:
        acl.append(['ac_'+ct,g,mean_expr_norm_ac.loc[ct,g],frac_ac.loc[ct,g]])
        
xtl =[]
for ct in mean_expr_norm_xt.index:
    for g in mean_expr_norm_xt.columns:
        xtl.append(['xt_'+ct,g,mean_expr_norm_xt.loc[ct,g],frac_xt.loc[ct,g]])
        
drl = []
for ct in mean_expr_norm_dr.index:
    for g in mean_expr_norm_dr.columns:
        drl.append(['dr_'+ct,g,mean_expr_norm_dr.loc[ct,g],frac_dr.loc[ct,g]])

In [95]:
cjdf = pd.DataFrame(data = cjl,columns = ['celltype','gene','avg exp','frac'])
acdf = pd.DataFrame(data = acl,columns = ['celltype','gene','avg exp','frac'])
xtdf = pd.DataFrame(data = xtl,columns = ['celltype','gene','avg exp','frac'])
drdf = pd.DataFrame(data = drl,columns = ['celltype','gene','avg exp','frac'])

In [96]:
df = pd.concat([cjdf,acdf,xtdf,drdf],axis = 0)

In [97]:
df = df.fillna(0)

In [98]:
df

,celltype,gene,avg exp,frac
0,cj_cj_ac_xt_dr_4,Lhx2,1.000000,0.225490
1,cj_cj_ac_xt_dr_4,Lhx9,1.000000,0.176471
2,cj_cj_ac_xt_dr_4,Tcf7l2,0.910246,0.985294
3,cj_cj_ac_xt_dr_4,Gata3,0.000000,0.000000
4,cj_cj_ac_xt_dr_4,Tal1,0.000000,0.000000
...,...,...,...,...
223,dr_ac_dr_1,Satb2,0.000000,0.000000
224,dr_ac_dr_1,Tfap2a,0.482559,0.002417
225,dr_ac_dr_1,Otx2,0.000000,0.000000
226,dr_ac_dr_1,Barhl2,1.000000,0.000806


In [99]:
fin_order =[]
for item in index_order:
    if 'cj' in item:
        fin_order.append('cj_' + item)
    if 'ac' in item:
        fin_order.append('ac_' + item)
    if 'xt' in item:
        fin_order.append('xt_' + item)
    if 'dr' in item:
        fin_order.append('dr_' + item)

In [100]:
fin_order

['cj_cj_ac_xt_dr_4',
 'ac_cj_ac_xt_dr_4',
 'xt_cj_ac_xt_dr_4',
 'dr_cj_ac_xt_dr_4',
 'cj_cj_ac_xt_dr_3',
 'ac_cj_ac_xt_dr_3',
 'xt_cj_ac_xt_dr_3',
 'dr_cj_ac_xt_dr_3',
 'cj_cj_ac_xt_dr_1',
 'ac_cj_ac_xt_dr_1',
 'xt_cj_ac_xt_dr_1',
 'dr_cj_ac_xt_dr_1',
 'cj_cj_xt_dr_2',
 'xt_cj_xt_dr_2',
 'dr_cj_xt_dr_2',
 'cj_cj_ac_xt_1',
 'ac_cj_ac_xt_1',
 'xt_cj_ac_xt_1',
 'cj_cj_xt_dr_1',
 'xt_cj_xt_dr_1',
 'dr_cj_xt_dr_1',
 'ac_ac_xt_3',
 'xt_ac_xt_3',
 'cj_cj_ac_1',
 'ac_cj_ac_1',
 'ac_ac_xt_4',
 'xt_ac_xt_4',
 'ac_ac_xt_2',
 'xt_ac_xt_2',
 'cj_cj_ac_2',
 'ac_cj_ac_2',
 'ac_ac_xt_1',
 'xt_ac_xt_1',
 'ac_ac_dr_1',
 'dr_ac_dr_1',
 'cj_cj_ac_3',
 'ac_cj_ac_3',
 'cj_cj_ac_4',
 'ac_cj_ac_4']

In [101]:
test = [0,.1,.25,.4]
fin_ct = []
fin_gene = []
avg_exp = []
frac = []
for th in test:
    for item in df['celltype'].unique():
        fin_ct.append(item)
        fin_gene.append('test_' + str(th))
        avg_exp.append(1)
        frac.append(th)

In [102]:
test_df = pd.DataFrame([fin_ct,fin_gene,avg_exp,frac], index = ['celltype','gene','avg exp','frac']).T

In [103]:
df = pd.concat([df,test_df])

In [104]:
df

,celltype,gene,avg exp,frac
0,cj_cj_ac_xt_dr_4,Lhx2,1.0,0.22549
1,cj_cj_ac_xt_dr_4,Lhx9,1.0,0.176471
2,cj_cj_ac_xt_dr_4,Tcf7l2,0.910246,0.985294
3,cj_cj_ac_xt_dr_4,Gata3,0.0,0.0
4,cj_cj_ac_xt_dr_4,Tal1,0.0,0.0
...,...,...,...,...
151,dr_cj_ac_xt_dr_3,test_0.4,1,0.4
152,dr_cj_ac_xt_dr_1,test_0.4,1,0.4
153,dr_cj_xt_dr_2,test_0.4,1,0.4
154,dr_cj_xt_dr_1,test_0.4,1,0.4


In [105]:
df['frac'] = df['frac'].astype(float)
df['avg exp'] = df['avg exp'].astype(float)

In [106]:
df['frac'] =  df['frac'].clip(upper=.4)

In [107]:
df[df['celltype'] == 'cj_cj_ac_xt_dr_4']

,celltype,gene,avg exp,frac
0,cj_cj_ac_xt_dr_4,Lhx2,1.000000,0.225490
1,cj_cj_ac_xt_dr_4,Lhx9,1.000000,0.176471
2,cj_cj_ac_xt_dr_4,Tcf7l2,0.910246,0.400000
3,cj_cj_ac_xt_dr_4,Gata3,0.000000,0.000000
4,cj_cj_ac_xt_dr_4,Tal1,0.000000,0.000000
5,cj_cj_ac_xt_dr_4,Ebf3,0.022609,0.024510
6,cj_cj_ac_xt_dr_4,Irx1,0.000000,0.000000
7,cj_cj_ac_xt_dr_4,Lef1,0.018697,0.039216
8,cj_cj_ac_xt_dr_4,Prdm16,0.054681,0.034314
9,cj_cj_ac_xt_dr_4,Zic1,0.023254,0.039216


In [108]:
mg_list

['Lhx2',
 'Lhx9',
 'Tcf7l2',
 'Gata3',
 'Tal1',
 'Ebf3',
 'Irx1',
 'Lef1',
 'Prdm16',
 'Zic1',
 'Zic4',
 'Pitx2',
 'Sim1',
 'Uncx',
 'Prdm13',
 'Sp5',
 'St18',
 'Casz1',
 'Esrrg',
 'Npas3',
 'Meis2',
 'Nr2f2',
 'Tshz2',
 'Bhlhe22',
 'Foxg1',
 'Tcf4',
 'Insm1',
 'Nr2e1',
 'Zfp516',
 'Prox1',
 'Nr3c2',
 'Rora',
 'Bsx',
 'Esr1',
 'Hmx2',
 'Neurod2',
 'Satb2',
 'Tfap2a',
 'Otx2',
 'Barhl2',
 'Zeb2']

In [110]:
fig = px.scatter(df, x = 'celltype', y = 'gene', size = 'frac', color = 'avg exp', color_continuous_scale= 'Blues', range_color=[0,1],opacity = 1,render_mode='pdf')
fig.update_xaxes(categoryorder='array', categoryarray= fin_order,
                range=[-0.5, len(df["celltype"].unique()) - 0.5])
fig.update_yaxes(categoryorder='array', categoryarray= mg_list + ['test_0','test_0.1','test_0.25','test_0.4'])
fig.update_layout(
    autosize=False,
    width=1800,
    height=1200,
)

max_size = 15

fig.update_traces(
    marker=dict(
        sizemode="area",
        sizeref=df["frac"].max() / max_size**2
    )
)

fig.update_xaxes(
    showgrid=False,
    showline=True,
    linewidth=1,
    linecolor="black",
    mirror=True,
)

fig.update_yaxes(
    showgrid=False,
    showline=True,
    linewidth=1,
    linecolor="black",
    mirror=True,
)
fig.update_layout(
    plot_bgcolor="white",
    paper_bgcolor="white",
)
fig.write_image("Figures/Figures_07242026/nonmammal_dotplot_test_TFexp_07232026.pdf")